# General Exploration

Set `FOLDER_NAME` below to point at any multi-run output folder.

In [ ]:
import sys
from pathlib import Path
_root = Path(".").resolve()
while not (_root / "paths.py").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))
import paths

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

FOLDER_NAME = FOLDER_NAME
run_folder = paths.MULTI_RUN_OUTPUT / FOLDER_NAME

df = pd.read_csv(run_folder / "results.csv")
print(f"Loaded {len(df)} runs  |  {df['scenario'].nunique()} scenarios  |  {df['policy'].nunique()} policies")

In [ ]:
phases = ["ccp_transport", "r1_care", "r1_transport", "r2_care"]
phase_labels = ["CCP Transport", "R1 Care", "R1 Transport", "R2 Care"]

POLICY_LABELS = {0: "strict", 1: "bn_pool", 2: "bde_pool"}
POLICIES = [
    (fe, ae)
    for fe in sorted(df["facility_eligibility"].unique())
    for ae in sorted(df["ambulance_eligibility"].unique())
    if not (fe == 0 and ae == 0)
]
strict = df[(df["facility_eligibility"] == 0) & (df["ambulance_eligibility"] == 0)]

x = np.arange(len(phases))
w = 0.35

## T1 — Patients started vs completed per phase

In [ ]:
started = [df[f"n_started_{p}_T1"].mean() for p in phases]
completed = []
for p in phases:
    col = f"n_completed_{p}_T1"
    if col in df.columns:
        completed.append(df[col].mean())
    elif f"n_safe_{p}_T1" in df.columns:
        completed.append(df[f"n_safe_{p}_T1"].mean())
    else:
        completed.append((df[f"n_started_{p}_T1"] * (1 - df[f"mortality_{p}_T1"])).mean())

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, started, w, label="Started", color="steelblue")
ax.bar(x + w/2, completed, w, label="Completed / Survived", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(phase_labels)
ax.set_ylabel("Mean number of patients")
ax.set_title("T1 — Patients started vs completed per phase")
ax.legend()
plt.tight_layout()
plt.show()

## T2 — Patients started vs completed per phase

In [ ]:
started = [df[f"n_started_{p}_T2"].mean() for p in phases]
completed = []
for p in phases:
    col = f"n_completed_{p}_T2"
    if col in df.columns:
        completed.append(df[col].mean())
    elif f"n_safe_{p}_T2" in df.columns:
        completed.append(df[f"n_safe_{p}_T2"].mean())
    else:
        completed.append((df[f"n_started_{p}_T2"] * (1 - df[f"mortality_{p}_T2"])).mean())

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, started, w, label="Started", color="steelblue")
ax.bar(x + w/2, completed, w, label="Completed / Survived", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(phase_labels)
ax.set_ylabel("Mean number of patients")
ax.set_title("T2 — Patients started vs completed per phase")
ax.legend()
plt.tight_layout()
plt.show()

## T1 — Median waiting time per phase

In [ ]:
waits = [df[f"median_wait_{p}_T1"].mean() for p in phases]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(phase_labels, waits, color="steelblue")
ax.set_ylabel("Mean of median wait (minutes)")
ax.set_title("T1 — Median waiting time per phase")
plt.tight_layout()
plt.show()

## T2 — Median waiting time per phase

In [ ]:
waits = [df[f"median_wait_{p}_T2"].mean() for p in phases]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(phase_labels, waits, color="coral")
ax.set_ylabel("Mean of median wait (minutes)")
ax.set_title("T2 — Median waiting time per phase")
plt.tight_layout()
plt.show()

## T1 — Median wait distributions: policy vs strict

In [ ]:
for fe, ae in POLICIES:
    plabel = f"fac={POLICY_LABELS[fe]}, amb={POLICY_LABELS[ae]}"
    pol = df[(df["facility_eligibility"] == fe) & (df["ambulance_eligibility"] == ae)]
    for p, phlabel in zip(phases, phase_labels):
        col = f"median_wait_{p}_T1"
        s = strict[col].dropna()
        q = pol[col].dropna()
        t, pval = stats.mannwhitneyu(s, q, alternative="greater")

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(s, bins=30, alpha=0.5, label="strict", color="steelblue")
        ax.hist(q, bins=30, alpha=0.5, label=plabel, color="coral")
        ax.set_xlabel("Median wait (minutes)")
        ax.set_ylabel("Scenarios")
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "n.s."
        ax.set_title(f"T1 — {phlabel}: {plabel} vs strict  |  p={pval:.4f} {sig}")
        ax.legend()
        plt.tight_layout()
        plt.show()

## T2 — Median wait distributions: policy vs strict

In [ ]:
for fe, ae in POLICIES:
    plabel = f"fac={POLICY_LABELS[fe]}, amb={POLICY_LABELS[ae]}"
    pol = df[(df["facility_eligibility"] == fe) & (df["ambulance_eligibility"] == ae)]
    for p, phlabel in zip(phases, phase_labels):
        col = f"median_wait_{p}_T2"
        s = strict[col].dropna()
        q = pol[col].dropna()
        t, pval = stats.mannwhitneyu(s, q, alternative="greater")

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(s, bins=30, alpha=0.5, label="strict", color="steelblue")
        ax.hist(q, bins=30, alpha=0.5, label=plabel, color="coral")
        ax.set_xlabel("Median wait (minutes)")
        ax.set_ylabel("Scenarios")
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "n.s."
        ax.set_title(f"T2 — {phlabel}: {plabel} vs strict  |  p={pval:.4f} {sig}")
        ax.legend()
        plt.tight_layout()
        plt.show()

## T1 — Mortality distributions: policy vs strict

In [ ]:
for fe, ae in POLICIES:
    plabel = f"fac={POLICY_LABELS[fe]}, amb={POLICY_LABELS[ae]}"
    pol = df[(df["facility_eligibility"] == fe) & (df["ambulance_eligibility"] == ae)]
    for p, phlabel in zip(phases, phase_labels):
        col = f"mortality_{p}_T1"
        s = strict[col].dropna()
        q = pol[col].dropna()
        t, pval = stats.mannwhitneyu(s, q, alternative="greater")

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(s, bins=30, alpha=0.5, label="strict", color="steelblue")
        ax.hist(q, bins=30, alpha=0.5, label=plabel, color="coral")
        ax.set_xlabel("Mortality rate")
        ax.set_ylabel("Scenarios")
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "n.s."
        ax.set_title(f"T1 — {phlabel}: {plabel} vs strict  |  p={pval:.4f} {sig}")
        ax.legend()
        plt.tight_layout()
        plt.show()

## T2 — Mortality distributions: policy vs strict

In [ ]:
for fe, ae in POLICIES:
    plabel = f"fac={POLICY_LABELS[fe]}, amb={POLICY_LABELS[ae]}"
    pol = df[(df["facility_eligibility"] == fe) & (df["ambulance_eligibility"] == ae)]
    for p, phlabel in zip(phases, phase_labels):
        col = f"mortality_{p}_T2"
        s = strict[col].dropna()
        q = pol[col].dropna()
        t, pval = stats.mannwhitneyu(s, q, alternative="greater")

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(s, bins=30, alpha=0.5, label="strict", color="steelblue")
        ax.hist(q, bins=30, alpha=0.5, label=plabel, color="coral")
        ax.set_xlabel("Mortality rate")
        ax.set_ylabel("Scenarios")
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "n.s."
        ax.set_title(f"T2 — {phlabel}: {plabel} vs strict  |  p={pval:.4f} {sig}")
        ax.legend()
        plt.tight_layout()
        plt.show()